# **Mini Project 2**

0. Requirements:
   
   If you do not have the following packages installed, run the command below to install them.

In [1]:
!pip install numpy
!pip install nltk
!pip install shap
!pip install matplotlib
!pip install tensorflow
!pip install scikit-learn
!pip install lime
!pip install codecarbon

In [2]:
import os
import re
import numpy as np
import nltk
import time
import shap
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
import tarfile
import urllib.request 
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from lime.lime_text import LimeTextExplainer
from sklearn.utils.class_weight import compute_class_weight
from codecarbon import EmissionsTracker
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score

2025-04-22 03:40:50.696101: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-22 03:40:50.696328: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-22 03:40:50.698647: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-22 03:40:50.727102: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-22 03:40:51.192875: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warnin

1. Data Preparation
   
    •	Goal: Load and inspect the IMDb dataset of labeled movie reviews.

    •	Task: Download the dataset (https://ai.stanford.edu/~amaas/data/sentiment/), and prepare labels (positive = 1, negative = 0) for training and test sets.

2. Text Preprocessing

    •	Goal: Clean and normalize raw text reviews.

    •	Task: Convert text to lowercase, remove HTML tags and punctuation, remove stopwords, and pad sequences to a fixed length (e.g., 200).


In [3]:
# Create a set of English stopwords for filtering
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
# On garde certains mots importants pour le sentiment
keep_words = {"not", "no", "nor", "don't", "didn't", "wasn't", "isn't", "weren't", "couldn't", "won't", "can't"}
# Retirer ces mots des stopwords
custom_stopwords = stop_words - keep_words

# Function to clean a single text string
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)               # Enlever les balises HTML
    text = re.sub(r"[^a-zA-Z']", " ", text)         # Garder seulement les lettres et apostrophes
    words = text.split()
    words = [w for w in words if w not in custom_stopwords]
    return " ".join(words)


# Nouvelle fonction pour comparer brut vs nettoyé
def preview_cleaning_example(directory):
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            with open(os.path.join(directory, filename), encoding="utf-8") as f:
                raw_text = f.read()
                cleaned_text = clean_text(raw_text)
                print(" Exemple de texte BRUT :\n")
                print(raw_text[:500])  # Affiche 500 caractères bruts
                print("\n Exemple de texte NETTOYÉ :\n")
                print(cleaned_text[:500])  # Même chose après nettoyage
                break  # On affiche seulement un exemple

#  Appelle cette fonction sur un exemple positif
preview_cleaning_example("aclImdb/train/pos")

# Function to load and clean reviews from a given directory
def load_reviews_from_dir(directory, label):
    texts, labels = [], []
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            with open(os.path.join(directory, filename), encoding="utf-8") as f:
                texts.append(clean_text(f.read()))
                labels.append(label)
    return texts, labels


# Load and preprocess training and testing data

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
archive_path = "aclImdb_v1.tar.gz"
data_dir = "aclImdb"

if not os.path.exists(data_dir):
    urllib.request.urlretrieve(url, archive_path)
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall()

# Charger les données positives et négatives
train_texts, train_labels = [], []
for label_type in ['pos', 'neg']:
    label = 1 if label_type == 'pos' else 0
    dir_path = os.path.join(data_dir, "train", label_type)
    texts, labels = load_reviews_from_dir(dir_path, label)
    train_texts.extend(texts)
    train_labels.extend(labels)

# Créer DataFrame
df = pd.DataFrame({'review': train_texts, 'sentiment': train_labels})

#  CHARGER  ENSEMBLE DE TEST (25 000 exemples)
test_texts, test_labels = [], []
for label_type in ['pos', 'neg']:
    label = 1 if label_type == 'pos' else 0
    dir_path = os.path.join(data_dir, "test", label_type)
    texts, labels = load_reviews_from_dir(dir_path, label)
    test_texts.extend(texts)
    test_labels.extend(labels)
    
train_df = pd.DataFrame({'review': train_texts, 'sentiment': train_labels})
test_df = pd.DataFrame({'review': test_texts, 'sentiment': test_labels})

#  Afficher les tailles pour vérification
print(f"Train set : {train_df.shape}")
print(f"Test set  : {test_df.shape}")

[nltk_data] Downloading package stopwords to /home/samira/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


 Exemple de texte BRUT :

I loves this movie,because it showed that they were not killing for fun but to save the ones they loved! Heath Ledger and Orlando Bloom did a great job portraying Ned and Joe. It has a few quick inappropriate scenes but is all right other than that. The language is very mild and sometimes don't even know it is there. This movie shows that just because they are outlaws does not mean that they are vicious killers! I hope that people will watch this movie and learn about important times in history 

 Exemple de texte NETTOYÉ :

loves movie showed not killing fun save ones loved heath ledger orlando bloom great job portraying ned joe quick inappropriate scenes right language mild sometimes don't even know movie shows outlaws not mean vicious killers hope people watch movie learn important times history like one one thing fascinates movie got inspiration armor book ned looked also people remember armor hope people watch movie get interested
Train set : (25000, 2)
T

3. Tokenization and Sequence Vectorization
   
    •	Goal: Convert preprocessed text into numerical format.
   
    •	Task: Use Keras Tokenizer to generate a word index and convert reviews into padded integer sequences.


In [4]:
# Create a tokenizer that will keep the top 12,000 most frequent words
vocab_size = 12000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(train_df['review'])

# Convert training and test texts into sequences of integers
X_train_seq = tokenizer.texts_to_sequences(train_df['review'])
X_test_seq = tokenizer.texts_to_sequences(test_df['review'])


# Pad or truncate the sequences so that all inputs have the same length (200 words)
maxlen = 200
X_train_padded = pad_sequences(X_train_seq, maxlen=maxlen, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=maxlen, padding='post', truncating='post')


# Convert labels to numpy arrays for training
train_df = pd.DataFrame({'review': train_texts, 'sentiment': train_labels})
test_df = pd.DataFrame({'review': test_texts, 'sentiment': test_labels})
y_train = np.array(train_df['sentiment'])
y_test = np.array(test_df['sentiment'])
print("Exemple de séquence encodée :", X_train_seq[0][:10])
print("Taille X_train :", X_train_padded.shape)
print("Taille y_train :", y_train.shape)


Exemple de séquence encodée : [1236, 1, 1028, 3, 740, 146, 477, 528, 328, 6972]
Taille X_train : (25000, 200)
Taille y_train : (25000,)


4. Model Design and Training

    •	Goal: Train a deep neural network using Bidirectional LSTM layers.
   
    •	Task:
   
        o	Use two stacked Bidirectional LSTM layers followed by Dense and Dropout.
   
        o	Use EarlyStopping to prevent overfitting.

In [5]:
# Define a sequential model for binary sentiment classification
model = Sequential([
    Embedding(input_dim=12000, output_dim=64),
    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(64)), # ou 32
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model with Adam optimizer and binary cross-entropy loss
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
# EarlyStopping
early_stop = EarlyStopping(
    monitor='loss',
    patience=3,
    restore_best_weights=True
)

# Entraînement du modèle
history = model.fit(
    X_train_padded, y_train,
    #validation_data=(X_val, y_val),#je peux l'enlever
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)
model.summary()


Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 56s 136ms/step - accuracy: 0.6979 - loss: 0.5393
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 54s 138ms/step - accuracy: 0.9214 - loss: 0.2256
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 57s 144ms/step - accuracy: 0.9564 - loss: 0.1337
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 62s 157ms/step - accuracy: 0.9739 - loss: 0.0835
Epoch 5/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 70s 178ms/step - accuracy: 0.9827 - loss: 0.0559
Epoch 6/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 74s 188ms/step - accuracy: 0.9842 - loss: 0.0504
Epoch 7/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 74s 189ms/step - accuracy: 0.9902 - loss: 0.0336
Epoch 8/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 74s 189ms/step - accuracy: 0.9927 - loss: 0.0269
Epoch 9/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 74s 189ms/step - accuracy: 0.9933 - loss: 0.0262
Epoch 10/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 74s 190ms/step - accuracy: 0.9950 - loss: 0.0167


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │       768,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200, 128)       │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,823,557 (10.77 MB)

 Trainable params: 941,185 (3.59 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,882,372 (7.18 MB)

In [ ]:
# Split the training data into training and validation sets (80/20 split)
X_train_final, X_val, y_train_final, y_val = train_test_split(
X_train_padded, y_train, test_size=0.2, random_state=42)

# Set up early stopping to prevent overfitting (stop if no improvement for 4 epochs)
early_stop = EarlyStopping(monitor='val_loss',patience=4,restore_best_weights=True,verbose=1)

# Initialize the CodeCarbon tracker to measure the carbon emissions during training
tracker = EmissionsTracker(project_name="IMDb_Sentiment_Model")
tracker.start()

# Train the model using the training data and validate on the validation set
history = model.fit(
    X_train_final, y_train_final,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# Stop the emissions tracker and capture the total carbon footprint
emissions = tracker.stop()


[codecarbon WARNING @ 03:52:04] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 03:52:04] [setup] RAM Tracking...
[codecarbon INFO @ 03:52:04] [setup] CPU Tracking...
[codecarbon WARNING @ 03:52:05] We saw that you have a 12th Gen Intel(R) Core(TM) i7-1255U but we don't know it. Please contact us.
[codecarbon WARNING @ 03:52:05] We will use the default power consumption of 4 W per thread for your 12 CPU, so 48W.
[codecarbon WARNING @ 03:52:05] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 03:52:05] CPU Model on constant consumption mode: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon WARNING @ 03:52:05] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon INFO @ 03:52:05] [setup] GPU Tracking...
[codecarbon INFO @ 03:52:05] No GPU found.
[codecarbon INFO @ 03:52:05] T

Epoch 1/10
 95/313 ━━━━━━━━━━━━━━━━━━━━ 34s 157ms/step - accuracy: 0.9964 - loss: 0.0207

[codecarbon INFO @ 03:52:25] Energy consumed for RAM : 0.000043 kWh. RAM Power : 10.0 W


 97/313 ━━━━━━━━━━━━━━━━━━━━ 34s 158ms/step - accuracy: 0.9965 - loss: 0.0206

[codecarbon INFO @ 03:52:25] Delta energy consumed for CPU with cpu_load : 0.000138 kWh, power : 32.02560000000001 W
[codecarbon INFO @ 03:52:25] Energy consumed for All CPU : 0.000138 kWh
[codecarbon INFO @ 03:52:25] 0.000181 kWh of electricity used since the beginning.


...

313/313 ━━━━━━━━━━━━━━━━━━━━ 56s 179ms/step - accuracy: 0.9984 - loss: 0.0067 - val_accuracy: 0.9738 - val_loss: 0.0911
Epoch 5/10
 60/313 ━━━━━━━━━━━━━━━━━━━━ 35s 142ms/step - accuracy: 0.9961 - loss: 0.0129

[codecarbon INFO @ 03:56:25] Energy consumed for RAM : 0.000687 kWh. RAM Power : 10.0 W


 64/313 ━━━━━━━━━━━━━━━━━━━━ 35s 142ms/step - accuracy: 0.9962 - loss: 0.0127

[codecarbon INFO @ 03:56:25] Delta energy consumed for CPU with cpu_load : 0.000111 kWh, power : 27.588 W
[codecarbon INFO @ 03:56:25] Energy consumed for All CPU : 0.002096 kWh
[codecarbon INFO @ 03:56:25] 0.002783 kWh of electricity used since the beginning.


123/313 ━━━━━━━━━━━━━━━━━━━━ 27s 143ms/step - accuracy: 0.9960 - loss: 0.0142

5. Carbon Footprint Analysis
   
    •	Goal: Track and report energy usage during training.
   
    •	Task: Use CodeCarbon to measure carbon emissions and display results in the notebook. You can find needed information here: https://mlco2.github.io/codecarbon/  &  https://codecarbon.io/


In [ ]:
# Display the total carbon emissions generated during model training
print(f"\n Emissions during model training: {emissions:} kgCO2eq")

6. Model Evaluation
   
    •	Goal: Assess the performance of your trained model.
   
    •	Task: Report training and validation accuracy/loss. Test the model on new review examples and interpret the predictions.


In [ ]:
# Evaluate the trained model on the test dataset to check final performance
test_loss, test_accuracy = model.evaluate(X_test_padded, y_test, verbose=1)

print(f"\n Test Accuracy: {test_accuracy:.4f}")
print(f" Test Loss: {test_loss:.4f}")


In [ ]:
# Predict the probabilities of the positive class for the test data
y_prob = model.predict(X_test_padded)
y_pred = (y_prob > 0.5).astype("int32")

# Confusion Matrix
# Générer la confusion matrix
cm = confusion_matrix(y_test, y_pred)
# Afficher graphiquement
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()

# Classification Report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))


In [ ]:
# Sample review texts to test sentiment predictions
sample_texts = [
    "The movie was absolutely fantastic and thrilling!",
    "The movie was so bad and boring!",
    "The movie had great visuals but the storyline was dull and predictable",
    "The movie had great visuals. It was a wonderful movie",
    "The plot was decent but the acting was terrible",
    "Nothing special, just an average movie",
    "An absolute masterpiece! Brilliant in every aspect",
    "Awful. Just awful. I walked out after 30 minutes"
]

# Function to predict sentiment for a list of input texts
def predict_sentiment(texts, tokenizer, model, maxlen=200):
    # Nettoyage
    cleaned = [clean_text(text) for text in texts]
    # Tokenisation
    sequences = tokenizer.texts_to_sequences(cleaned)
    padded = pad_sequences(sequences, maxlen=maxlen, padding='post')
    # Prédictions
    probs = model.predict(padded)
    # Affichage des résultats
    for i, text in enumerate(texts):
        sentiment = "Positive ^-^" if probs[i] > 0.7 else "Negative -_-"
        print(f" Review: {text}\n→ Prediction: {sentiment} (score: {probs[i][0]:.2f})\n")

# Run the sentiment prediction function on the sample texts
predict_sentiment(sample_texts, tokenizer, model)

7. Explainability with SHAP and LIME
   
    •	Goal: Visualize and explain individual predictions.
   
    •	Task: Use SHAP and LIME to analyze the impact of specific words on sentiment classification and interpret model behavior.


In [ ]:
# A smaller set of sample texts for explanation analysis
sample_texts2 = [
    "The movie was so bad and boring!",
    "The movie had great visuals. It was a wonderful movie",
]

# Preprocess and pad the input text for model compatibility
sample_sequences2 = tokenizer.texts_to_sequences(sample_texts2)
sample_padded2 = pad_sequences(sample_sequences2, maxlen=200, padding='post')


# ---------- LIME ----------
print("LIME Explanation Results :\n")

explainer = LimeTextExplainer(class_names=["Negative", "Positive"])

# Fonction de prédiction adaptée pour LIME
def predict_fn(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(sequences, maxlen=200, padding='post')
    predictions = model.predict(padded)
    proba_neg = 1 - predictions
    return np.hstack((proba_neg, predictions))

# Afficher les explications LIME pour chaque texte
for i, text in enumerate(sample_texts2):
    print(f"\n LIME Explanation for: \"{text}\"")
    explanation = explainer.explain_instance(
        text_instance=text,
        classifier_fn=predict_fn,
        num_features=10
    )
    from IPython.display import display, HTML
    html = explanation.as_html()
    display(HTML(html))

    
# ---------- SHAP ----------
print("SHAP Explanation Results: \n")

# Fond de référence
X_background = X_train_padded[np.random.choice(X_train_padded.shape[0], 100, replace=False)]
# Initialiser SHAP
explainer_shap = shap.KernelExplainer(model.predict, X_background)

# Calculer SHAP values pour les 2 phrases
shap_values = explainer_shap.shap_values(sample_padded2, nsamples=100)
# Vérifier les dimensions
print("shap_values shape:", np.array(shap_values).shape)
print("sample_padded2 shape:", sample_padded2.shape)

##PREMIERE METHODE   -- que 2 mots
## Conversion des indices en mots (on utilise uniquement la première phrase ici)
#feature_names = np.array([tokenizer.index_word.get(idx, 'PAD') for idx in sample_padded2[0]])
## Identifier seulement les mots réels (indices != 0)
#non_pad_indices = sample_padded2[0] != 0
## SHAP values correspondantes (première phrase uniquement)
#shap_values_non_pad = np.squeeze(shap_values)[0][non_pad_indices]
##Mots réels correspondants
#feature_names_non_pad = feature_names[non_pad_indices]
## Features pour SHAP (doit être 2D)
#features_non_pad = sample_padded2[0][non_pad_indices].reshape(1, -1)
## Afficher correctement le SHAP summary_plot
#shap.summary_plot(
#    shap_values_non_pad.reshape(1, -1),
#    features=features_non_pad,
#    feature_names=feature_names_non_pad,
#    plot_type="bar"
#)



##DEUXIEME METHODE   -- que 3 mots
index_to_word = tokenizer.index_word  

tokens_to_words = []
for sequence in sample_padded2:
    words = [index_to_word.get(idx, str(idx)) for idx in sequence]  
    tokens_to_words.append(words)

feature_names = tokens_to_words[0] 

shap.summary_plot(
    np.squeeze(shap_values),
    sample_padded2,
    feature_names=feature_names
)


##TROISIEME METHODE   -- que 3 mots
#Dictionnaire index → mot
#index_to_word = tokenizer.index_word

# Séquence de la première phrase
#sequence = sample_padded2[0]
#shap_vals = np.squeeze(shap_values)[0]

# Garder uniquement les positions avec des vrais mots (non 0)
#non_zero_indices = sequence != 0

# Extraire les indices et leurs mots correspondants
#words = [index_to_word.get(idx, f'UNK_{idx}') for idx in sequence[non_zero_indices]]
#values = shap_vals[non_zero_indices]
#features = sequence[non_zero_indices].reshape(1, -1)

# Afficher le SHAP summary plot
#shap.summary_plot(
#    values.reshape(1, -1),
#    features,
#    feature_names=words,
#    plot_type="bar"
#)


8. Deployment Considerations
    
    •	Goal: Optimize the model for embedded or mobile environments.
    
    •	Task: Convert the model to TensorFlow Lite and compare:
    
        o	Model size
    
        o	Inference time
    
        o	Accuracy difference before/after conversion


In [ ]:
# Create a TFLite converter from the trained Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

# Désactiver le lowering des TensorList ops (nécessaire pour LSTM/Bidirectional)
converter._experimental_lower_tensor_list_ops = False

# Autoriser les variables de ressources (comme les états internes du LSTM)
converter.experimental_enable_resource_variables = True


# Save the converted model to a .tflite file
tflite_model = converter.convert()
with open("model.tflite", "wb") as f:
    f.write(tflite_model)

In [ ]:
# Load the TFLite model using TensorFlow Lite Interpreter
interpreter = tf.lite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

# Get input and output tensor details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Run a single inference to test functionality and measure speed
test_sample = X_test_padded[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_sample)

# Measure inference time for a single sample
start_time = time.time()
interpreter.invoke()
end_time = time.time()

# Retrieve the output of the model
output = interpreter.get_tensor(output_details[0]['index'])
print("Prediction:", output[0][0])
print(" Inference time :", round((end_time - start_time) * 1000, 2), "ms")


# Calculate the size of the .tflite model file
model_size_kb = os.path.getsize("model.tflite") / 1024
print(f" TFLite model size: {model_size_kb:.2f} KB")

# Run inference on the full test set using the TFLite model
correct = 0
total = len(X_test_padded)

for i in range(total):
    input_data = np.expand_dims(X_test_padded[i], axis=0).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    prediction = interpreter.get_tensor(output_details[0]['index'])[0][0]
    predicted_label = int(prediction > 0.5)
    if predicted_label == y_test[i]:
        correct += 1

    
# Evaluate the TFLite model's accuracy
accuracy = correct / total
print(f"TFLite model accuracy : {accuracy:.4f} ({correct}/{total})")
